# 🌱 AI-Based Irrigation Optimizer v4 — Larger Real Data + Blend
**"Right Water. Right Time. Right Crop."**

**What changed vs v3:**
- ❌ Dropped `agriculture-and-farming-dataset` (50 rows) and `fertilizer-prediction` (~99 rows) — too small to matter.
- ✅ Added **NASA POWER API** — free, no signup, no key, real daily weather history for any location/date range (effectively unlimited rows).
- ✅ Added **Kaggle Crop Recommendation Dataset** (`atharvaingle/crop-recommendation-dataset`) — 2,200 real rows (N/P/K, temp, humidity, rainfall, crop label).
- ✅ Kept Mendeley + Zindi real sensor logs from v3.
- ✅ **No public dataset actually has a labeled "irrigation mm required" target** — that's a derived/simulated quantity in every research paper too. So real feature data (weather, soil, crop) is combined with the physics-based water-balance formula for the target label, then optionally padded with pure synthetic rows to reach a solid training size. This is standard practice (it's literally how the ICAR/FAO methodology works — physics model, not a labeled dataset).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings, os, glob, requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
os.makedirs('data', exist_ok=True)
print("Libraries loaded.")

## 2. Get the Datasets

### A. NASA POWER API — real weather, no signup (pulled live below)
No download needed — the notebook calls it directly:
`https://power.larc.nasa.gov/api/temporal/daily/point`

### B. Kaggle Crop Recommendation Dataset (2,200 rows)
```python
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d atharvaingle/crop-recommendation-dataset -p data --unzip
```
File expected: `data/Crop_recommendation.csv`

### C. Mendeley (no login) — https://data.mendeley.com/datasets/fpdwmm7nrb/1
Download zip → extract → save as `data/mendeley_soil_motor.csv`

### D. Zindi (free account) — WaziHub Soil Moisture Challenge
Download `Train.csv` → save as `data/zindi_train.csv`

Run whatever you have — each loader below is independent and skipped if its file is missing.

In [ ]:
KC_TABLE = {
    'Rice':      {'Initial': 1.05, 'Development': 1.10, 'Mid-season': 1.20, 'Late-season': 0.90},
    'Wheat':     {'Initial': 0.40, 'Development': 0.75, 'Mid-season': 1.15, 'Late-season': 0.40},
    'Cotton':    {'Initial': 0.35, 'Development': 0.70, 'Mid-season': 1.15, 'Late-season': 0.65},
    'Sugarcane': {'Initial': 0.40, 'Development': 0.85, 'Mid-season': 1.25, 'Late-season': 0.75},
    'Maize':     {'Initial': 0.30, 'Development': 0.70, 'Mid-season': 1.20, 'Late-season': 0.60},
    'Coffee':    {'Initial': 0.90, 'Development': 0.95, 'Mid-season': 1.05, 'Late-season': 0.95},
    'Chickpea':  {'Initial': 0.40, 'Development': 0.70, 'Mid-season': 1.00, 'Late-season': 0.35},
    'Banana':    {'Initial': 0.50, 'Development': 0.90, 'Mid-season': 1.10, 'Late-season': 1.00},
}
ROOT_ZONE_DEPTH_MM = {'Rice': 400, 'Wheat': 900, 'Cotton': 1200, 'Sugarcane': 1500,
                      'Maize': 1000, 'Coffee': 1200, 'Chickpea': 700, 'Banana': 900}
CROP_SYNONYMS = {'paddy': 'Rice', 'rice': 'Rice', 'wheat': 'Wheat', 'cotton': 'Cotton',
                  'sugarcane': 'Sugarcane', 'maize': 'Maize', 'corn': 'Maize',
                  'coffee': 'Coffee', 'chickpea': 'Chickpea', 'banana': 'Banana'}
SOIL_SYNONYMS = {'alluvial': 'Loamy', 'sandy': 'Sandy', 'loamy': 'Loamy',
                  'clayey': 'Clayey', 'clay': 'Clayey', 'silty': 'Loamy', 'peaty': 'Loamy'}
DEFAULT_KC, DEFAULT_ROOT_DEPTH = 0.85, 900
UNIFIED_COLS = ['temperature_c','humidity_pct','wind_speed_kmph','rainfall_forecast_mm',
                'soil_moisture_pct','field_capacity_pct','et0_mm','crop_type','soil_type',
                'growth_stage','irrigation_required_mm']

def normalize_crop(name):
    return CROP_SYNONYMS.get(str(name).strip().lower(), name if name in KC_TABLE else 'Maize')

def normalize_soil(name):
    return SOIL_SYNONYMS.get(str(name).strip().lower(), name if name in ['Sandy','Loamy','Clayey'] else 'Loamy')

def estimate_et0(temp, humidity, solar=18.0):
    return np.clip(0.0023 * (solar * 30) * (np.asarray(temp) + 17.8) * (1 - np.asarray(humidity) / 100) ** 0.5, 1, 12)

def synth_target(temp, humidity, rainfall, soil_moist, field_cap, crop, stage):
    kc = np.array([KC_TABLE.get(c, {}).get(s, DEFAULT_KC) for c, s in zip(crop, stage)])
    et0 = estimate_et0(temp, humidity)
    deficit = np.clip(np.asarray(field_cap) - np.asarray(soil_moist), 0, None)
    return np.clip(kc * et0 + 0.1 * deficit - np.asarray(rainfall) * 0.75, 0, None)

print("Reference tables + normalizers loaded.")

## 3. Loaders — Real Data Sources

In [ ]:
def load_nasa_power(lat=28.6, lon=77.2, start='20200101', end='20231231'):
    """Real daily weather from NASA POWER — free, no key. Default point: Delhi region."""
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point"
           f"?parameters=T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN,PRECTOTCORR"
           f"&community=AG&longitude={lon}&latitude={lat}&start={start}&end={end}&format=JSON")
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        p = r.json()['properties']['parameter']
        dates = list(p['T2M'].keys())
        df = pd.DataFrame({
            'temperature_c': [p['T2M'][d] for d in dates],
            'humidity_pct': [p['RH2M'][d] for d in dates],
            'wind_speed_kmph': [p['WS2M'][d] * 3.6 for d in dates],
            'rainfall_forecast_mm': [p['PRECTOTCORR'][d] for d in dates],
            'solar_mj': [p['ALLSKY_SFC_SW_DWN'][d] for d in dates],
        })
        df = df[(df['temperature_c'] > -50)]  # NASA uses -999 for missing
        return df
    except Exception as e:
        print(f"NASA POWER fetch failed: {e}")
        return None

def load_crop_recommendation():
    """Kaggle: atharvaingle/crop-recommendation-dataset (2200 real rows)"""
    path = 'data/Crop_recommendation.csv'
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    out = pd.DataFrame()
    out['temperature_c'] = df['temperature']
    out['humidity_pct'] = df['humidity']
    out['rainfall_forecast_mm'] = df['rainfall'] / 30.0  # annual-ish -> daily-ish scale
    out['crop_type'] = df['label'].apply(normalize_crop)
    return out

def load_mendeley_motor():
    matches = glob.glob('data/mendeley_soil_motor*.csv') or glob.glob('data/*motor*.csv')
    if not matches:
        return None
    df = pd.read_csv(matches[0])
    df.columns = [c.strip().lower() for c in df.columns]
    out = pd.DataFrame()
    out['soil_moisture_pct'] = df.get('soil_moisture', df.get('moisture'))
    out['temperature_c'] = df.get('temperature', df.get('air_temperature'))
    out['humidity_pct'] = df.get('humidity')
    return out.dropna(subset=['soil_moisture_pct']) if 'soil_moisture_pct' in out else None

def load_zindi_wazihub():
    path = 'data/zindi_train.csv'
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    out = pd.DataFrame()
    out['soil_moisture_pct'] = df.get('soil_humidity', df.get('humidity_soil'))
    out['temperature_c'] = df.get('air_temperature')
    out['humidity_pct'] = df.get('air_humidity')
    return out.dropna(subset=['soil_moisture_pct']) if 'soil_moisture_pct' in out else None

print("Loaders defined.")

## 4. Build the Combined Real+Synthetic Training Set
Pulls every available real source, fills schema gaps (crop/soil/growth-stage/soil-moisture aren't in weather data, and vice versa) via **random pairing from real value pools** rather than pure noise, then computes the physics-based target and tops up with pure synthetic rows to hit a solid training size.

In [ ]:
TARGET_ROWS = 8000

real_weather = load_nasa_power()
real_crop = load_crop_recommendation()
real_soil_sources = [d for d in [load_mendeley_motor(), load_zindi_wazihub()] if d is not None]
real_soil = pd.concat(real_soil_sources, ignore_index=True) if real_soil_sources else None

for name, d in [('NASA POWER weather', real_weather), ('Crop Recommendation', real_crop), ('Soil sensor logs', real_soil)]:
    print(f"{name}: {'not available' if d is None else f'{len(d)} real rows'}")

crops = list(KC_TABLE.keys())
soils = ['Sandy', 'Loamy', 'Clayey']
stages = ['Initial', 'Development', 'Mid-season', 'Late-season']

N = TARGET_ROWS
def pool(series, n, fallback):
    if series is not None and len(series.dropna()) > 0:
        return series.dropna().sample(n, replace=True).reset_index(drop=True)
    return pd.Series(np.random.choice(fallback, n) if isinstance(fallback, list)
                      else np.random.uniform(*fallback, n))

df = pd.DataFrame()
df['temperature_c'] = pool(real_weather['temperature_c'] if real_weather is not None else None, N, (10, 48))
df['humidity_pct'] = pool(real_weather['humidity_pct'] if real_weather is not None else None, N, (10, 95))
df['wind_speed_kmph'] = pool(real_weather['wind_speed_kmph'] if real_weather is not None else None, N, (0, 40))
df['rainfall_forecast_mm'] = pool(real_weather['rainfall_forecast_mm'] if real_weather is not None else None, N, (0, 26))
df['soil_moisture_pct'] = pool(real_soil['soil_moisture_pct'] if real_soil is not None else None, N, (5, 45))
df['field_capacity_pct'] = (df['soil_moisture_pct'] + np.random.uniform(5, 20, N)).clip(upper=45)
df['crop_type'] = pool(real_crop['crop_type'] if real_crop is not None else None, N, crops)
df['soil_type'] = np.random.choice(soils, N)
df['growth_stage'] = np.random.choice(stages, N)
df['et0_mm'] = estimate_et0(df['temperature_c'], df['humidity_pct'])
df['irrigation_required_mm'] = synth_target(df['temperature_c'], df['humidity_pct'], df['rainfall_forecast_mm'],
                                             df['soil_moisture_pct'], df['field_capacity_pct'],
                                             df['crop_type'], df['growth_stage'])
print(f"\nFinal training set: {df.shape}")
df.head()

## 5. Feature Engineering & Train/Test Split

In [ ]:
df['moisture_deficit_pct'] = (df['field_capacity_pct'] - df['soil_moisture_pct']).clip(lower=0)
df['rain_adjusted_et0'] = (df['et0_mm'] - df['rainfall_forecast_mm'] * 0.1).clip(lower=0)
df['kc'] = df.apply(lambda r: KC_TABLE.get(r['crop_type'], {}).get(r['growth_stage'], DEFAULT_KC), axis=1)
df['root_zone_depth_mm'] = df['crop_type'].map(ROOT_ZONE_DEPTH_MM).fillna(DEFAULT_ROOT_DEPTH)

feature_cols_num = ['temperature_c','humidity_pct','wind_speed_kmph','rainfall_forecast_mm',
                     'soil_moisture_pct','field_capacity_pct','et0_mm','kc','root_zone_depth_mm',
                     'moisture_deficit_pct','rain_adjusted_et0']
feature_cols_cat = ['crop_type','soil_type','growth_stage']

X = df[feature_cols_num + feature_cols_cat]
y = df['irrigation_required_mm']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), feature_cols_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
])
print("Train/test split:", X_train.shape, X_test.shape)

## 6. Train & Compare Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
}

results, pipelines = {}, {}
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = {'RMSE': mean_squared_error(y_test, preds) ** 0.5,
                      'MAE': mean_absolute_error(y_test, preds),
                      'R2': r2_score(y_test, preds)}
    pipelines[name] = pipe

results_df = pd.DataFrame(results).T.sort_values('RMSE')
best_model_name = results_df.index[0]
best_pipeline = pipelines[best_model_name]
print(results_df)
print(f"\nBest model: {best_model_name}")

## 7. Decision Engine (FlowState spec)
🟢 LOW / 🟡 MODERATE / 🔴 HIGH tiers, confidence %, irrigation duration. **Auto-normalizes crop/soil synonyms** (e.g. `'Paddy'`→`'Rice'`, `'Alluvial'`→`'Loamy'`) so unknown labels don't silently zero out.

In [ ]:
def get_confidence(pipeline, input_df):
    model = pipeline.named_steps['model']
    if hasattr(model, 'estimators_'):
        X_t = pipeline.named_steps['preprocessor'].transform(input_df)
        trees = np.array(model.estimators_).ravel()
        tree_preds = np.array([t.predict(X_t)[0] for t in trees])
        if tree_preds.mean() > 0:
            spread = tree_preds.std() / (abs(tree_preds.mean()) + 1e-6)
            return float(np.clip(100 - spread * 100, 40, 99))
    return 75.0

def recommend_irrigation(pipeline, reading: dict, field_area_m2: float = 1000.0, flow_rate_lpm: float = 200.0):
    r = dict(reading)
    r['crop_type'] = normalize_crop(r['crop_type'])
    r['soil_type'] = normalize_soil(r['soil_type'])
    r['kc'] = KC_TABLE.get(r['crop_type'], {}).get(r['growth_stage'], DEFAULT_KC)
    r['root_zone_depth_mm'] = ROOT_ZONE_DEPTH_MM.get(r['crop_type'], DEFAULT_ROOT_DEPTH)
    r['moisture_deficit_pct'] = max(0, r['field_capacity_pct'] - r['soil_moisture_pct'])
    r['rain_adjusted_et0'] = max(0, r['et0_mm'] - r['rainfall_forecast_mm'] * 0.1)

    input_df = pd.DataFrame([r])[feature_cols_num + feature_cols_cat]
    predicted_mm = float(pipeline.predict(input_df)[0])
    confidence = get_confidence(pipeline, input_df)

    if r['soil_moisture_pct'] >= r['field_capacity_pct']:
        tier, level, reason = "NO", "LOW", "Soil already at/above field capacity — waterlogging risk."
        final_mm = 0.0
    elif r['rainfall_forecast_mm'] > 10:
        tier, level, reason = "NO", "LOW", f"Significant rain forecast ({r['rainfall_forecast_mm']} mm) — wait and re-check."
        final_mm = 0.0
    elif predicted_mm < 1.0:
        tier, level, reason = "NO", "LOW", "Predicted need is negligible (<1 mm)."
        final_mm = 0.0
    elif predicted_mm < 4.0:
        tier, level, reason = "YES", "MODERATE", "Moderate water deficit for current crop stage/conditions."
        final_mm = round(predicted_mm, 2)
    else:
        tier, level, reason = "YES", "HIGH", "High water deficit — irrigate soon to avoid crop stress."
        final_mm = round(predicted_mm, 2)

    liters = round(final_mm * field_area_m2, 1)
    minutes = round(liters / flow_rate_lpm, 1) if liters > 0 else 0.0

    return {
        "irrigation_required": tier, "need_level": level, "reason": reason,
        "raw_model_prediction_mm": round(predicted_mm, 2),
        "recommended_water_liters": liters, "recommended_time_minutes": minutes,
        "confidence_pct": round(confidence, 1),
    }

## 8. Save Model

In [ ]:
os.makedirs('artifacts', exist_ok=True)
model_path = 'artifacts/irrigation_model.joblib'
joblib.dump(best_pipeline, model_path)
print(f"Saved '{best_model_name}' to {model_path}")
# from google.colab import files; files.download(model_path)

## 9. Demo

In [ ]:
sample_reading = {
    'temperature_c': 42, 'humidity_pct': 30, 'wind_speed_kmph': 4,
    'rainfall_forecast_mm': 0, 'soil_moisture_pct': 20, 'field_capacity_pct': 33,
    'et0_mm': 8.6, 'crop_type': 'Paddy', 'soil_type': 'Alluvial', 'growth_stage': 'Mid-season',
}
result = recommend_irrigation(best_pipeline, sample_reading, field_area_m2=2000, flow_rate_lpm=200)
for k, v in result.items():
    print(f"{k:28s}: {v}")

## 10. Dataset Reference

| Source | Real rows | Provides | Setup |
|---|---|---|---|
| NASA POWER API | Unlimited (any date range) | temp, humidity, wind, rainfall | none — auto-fetched |
| Kaggle Crop Recommendation | 2,200 | crop type, temp, humidity, rainfall | Kaggle API |
| Mendeley soil/motor logs | Varies (real sensor logs) | soil moisture, temp, humidity | manual download, no login |
| Zindi WaziHub | Varies (real sensor logs) | soil moisture, temp | free account |

**Where downloads land in Colab:** `data/` = `/content/data/` — check the folder icon in the left sidebar, or run `!ls data/`.

**Why blend with synthetic:** no public dataset labels "how many mm of irrigation was actually needed" — that number always comes from a water-balance calculation (ETc = Kc × ET0 − effective rainfall), which is exactly what agronomy research and products like this use. Real data here supplies realistic feature *distributions*; the label still comes from the physics formula, same as v1–v3, just anchored to real inputs now instead of pure noise.